# 🧪 StockGen AI - Custom Local Sandbox Notebook (Real-ESRGAN + GFPGAN)
ยินดีต้อนรับสู่ห้องทดลองจำลองระบบอัปสเกลระดับพรีเมียม! หน้าที่ของ Sandbox Notebook ตัวนี้คือ **ทดสอบและวิเคราะห์การทำงานร่วมกันระหว่างโมเดลขยายพิกเซล (Real-ESRGAN) และโมเดลปรับรายละเอียดใบหน้า (GFPGAN)** 

พร้อมทั้งจำลองการรับส่งข้อมูล (API Payload Mocking) ในรูปแบบเดียวกับ **RunPod Serverless** 100% ก่อนนำไปบิวด์แพ็คเกจ Docker และ Deploy ขึ้นระบบคลาวด์จริง เพื่อให้มั่นใจในคุณภาพ ความเสถียร และความคมชัดสูงสุดของทุกผลงานครับ! 🔬🎨

> [!IMPORTANT]
> **วิธีการรันใช้งาน:** กรุณากดปุ่มรัน (Play 🔴) เซลล์ด้านล่างเรียงตามลำดับตั้งแต่ **สเต็ปที่ 1 ถึง 4** เสมอครับ เพื่อติดตั้งไลบรารีและโหลดโมเดลเตรียมความพร้อมใช้งานได้อย่างราบรื่น

### 📦 สเต็ปที่ 1: ติดตั้งคลังไลบรารีและแก้จุดบกพร่องความเข้ากันได้ (Compatibility Auto-Patch)
**สำคัญมาก:** หากรันครั้งแรกบน Google Colab หรือเครื่องโลคอลที่ยังไม่มีคลังไลบรารี กรุณากดรันเซลล์นี้เพื่อติดตั้งโปรแกรมที่จำเป็นทั้งหมด พร้อมประยุกต์ใช้ Patch ความเข้ากันได้สำหรับ basicsr อัตโนมัติครับ!

In [ ]:
#@title ⚙️ 1) ติดตั้งไลบรารีและอิมพอร์ต Patch ความเข้ากันได้ { display-mode: "form" }
import sys
import os
import types

print("⏳ กำลังตรวจสอบและติดตั้งคลังโปรแกรมไลบรารี...\n")
# สั่งติดตั้ง Dependencies จริง (เปิดใช้งานออโต้บน Colab ทันที!)
!pip install realesrgan gfpgan opencv-python matplotlib pillow tqdm

print("\n🩹 กำลังประมวลผลแทรก Compatibility Patch ป้องกันข้อผิดพลาด basicsr...")
# ─── 🩹 แพตช์เพื่อความเข้ากันได้ของ basicsr กับ PyTorch เวอร์ชันใหม่ ───
try:
    import torchvision.transforms.functional as F_tv
    shim = types.ModuleType("torchvision.transforms.functional_tensor")
    shim.rgb_to_grayscale = F_tv.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = shim
    print("✅ สมัครและแทรก Compatibility Patch ให้ basicsr สำเร็จ!")
except Exception as e:
    print(f"⚠️ ข้อผิดพลาดในขั้นตอน Patch (หากรันเป็นครั้งแรกกรุณาข้ามได้): {str(e)}")

print("🎉 ระบบการ Patch และไลบรารีพร้อมประมวลผลระดับฐานโครงสร้าง!")

### 🧠 สเต็ปที่ 2: ฟังก์ชันการอัปสเกลพรีเมียมและการเตรียมโมเดล AI
รันเซลล์ด้านล่างเพื่อกำหนดรูปแบบโมเดลทั้ง 4 แบบที่ใช้งาน และจัดสร้างฟังก์ชันโหลดน้ำหนักโมเดลอย่างเป็นระเบียบ

In [ ]:
#@title ⚙️ 2) สร้างตัวโหลดและคอนฟิกโมเดล AI (Real-ESRGAN & GFPGAN) { display-mode: "form" }
import torch
import urllib.request
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from gfpgan import GFPGANer

# คอนฟิกการทำงานของโมเดลขยายพิกเซลหลัก
MODELS = {
    "x2plus": {
        "url": "https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth",
        "file": "RealESRGAN_x2plus.pth",
        "scale": 2,
        "block": 23
    },
    "x4plus": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "file": "RealESRGAN_x4plus.pth",
        "scale": 4,
        "block": 23
    },
    "anime": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
        "file": "RealESRGAN_x4plus_anime_6B.pth",
        "scale": 4,
        "block": 6
    },
    "ultrasharp": {
        "url": "https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth",
        "file": "4x-UltraSharp.pth",
        "scale": 4,
        "block": 23
    }
}

weights_dir = "./weights"
os.makedirs(weights_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'🖥️ อุปกรณ์คำนวณที่เครื่องตรวจพบ: {device.type.upper()}' + (f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''))

def get_upsampler(model_key):
    """โหลดโมเดล Real-ESRGAN แบบไดนามิกอิงตาม Model Key ที่ผู้ใช้เลือก"""
    cfg = MODELS[model_key]
    model_path = os.path.join(weights_dir, cfg["file"])
    
    if not os.path.exists(model_path):
        print(f"📥 กำลังดาวน์โหลดน้ำหนักโมเดล {cfg['file']} (~60-100MB)...")
        urllib.request.urlretrieve(cfg["url"], model_path)
        print("✅ ดาวน์โหลดเสร็จสิ้น!")
        
    # ─── 🩹 แก้ไข KeyError: 'params' สำหรับโมเดลชุมชนอย่าง 4x-UltraSharp ───
    try:
        loadnet = torch.load(model_path, map_location='cpu')
        if 'params' not in loadnet and 'params_ema' not in loadnet:
            print("🩹 ตรวจพบโครงสร้างโมเดลคอมมูนิตี้ แปลงโครงสร้างให้เข้ากับ RealESRGANer...")
            torch.save({'params': loadnet}, model_path)
            print("✅ ปรับปรุงและบันทึกน้ำหนักโมเดลใหม่สำเร็จ!")
    except Exception as e:
        print(f"⚠️ ข้อควรระวังระหว่างตรวจสอบน้ำหนักโมเดล: {str(e)}")
        
    # โครงสร้างโครงข่ายประสาทเทียม RRDBNet
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=cfg["block"], num_grow_ch=32, scale=cfg["scale"])
    upsampler = RealESRGANer(
        scale=cfg["scale"],
        model_path=model_path,
        model=model,
        tile=800,  # แบ่งกระเบื้องประมวลผลขนาด 800px ป้องกัน GPU แฮงก์
        tile_pad=10,
        pre_pad=10,
        half=device.type == "cuda",
        device=device
    )
    return upsampler, cfg["scale"]

def get_face_enhancer(upsampler, scale):
    """โหลดระบบกู้ใบหน้า GFPGANv1.3 และผูกมิตรกับเครื่องมือขยายพื้นหลัง"""
    gfpgan_path = os.path.join(weights_dir, "GFPGANv1.3.pth")
    if not os.path.exists(gfpgan_path):
        print("📥 กำลังดาวน์โหลดน้ำหนักระบบแต่งหน้า GFPGANv1.3 (~300MB)...")
        urllib.request.urlretrieve(
            "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth",
            gfpgan_path
        )
        print("✅ ดาวน์โหลดระบบแต่งใบหน้าสำเร็จ!")
        
    face_enhancer = GFPGANer(
        model_path=gfpgan_path,
        upscale=scale,
        arch="clean",
        channel_multiplier=2,
        bg_upsampler=upsampler
    )
    return face_enhancer

print("🚀 คลังฟังก์ชันเตรียมโมเดล AI พร้อมสแตนด์บายการใช้งานแล้ว!")

### 🧪 สเต็ปที่ 3: ตัวจำลอง RunPod Serverless API Handler (Mock API)
รันเซลล์นี้เพื่อเปิดเครื่องมือประมวลผลจำลอง ซึ่งจะรับ JSON Payload แบบเดียวกับที่เว็บ Next.js ส่งเข้าหา RunPod ในชีวิตจริง และแปลงภาพตอบกลับเป็น JSON Base64 ในรูปแบบที่เป็นมาตรฐาน 100%!

In [ ]:
#@title ⚙️ 3) เครื่องมือประมวลผลจำลอง RunPod Serverless API { display-mode: "form" }
import cv2
import base64
import tempfile
import numpy as np

def runpod_mock_handler(job_payload):
    """
    จำลองการกระทำตัวเป็น RunPod Serverless Handler ทำหน้าที่แกะ Payload ดำเนินงาน AI
    และตอบกลับข้อมูลผลลัพธ์ออกมาในลักษณะโครงสร้าง JSON ของจริง
    """
    print("⚡ [RunPod Mock SDK] เริ่มรับ Job และวิเคราะห์ Input Payload...")
    job_input = job_payload.get("input", {})
    image_source = job_input.get("image")
    model_name = job_input.get("model", "x2plus")
    face_enhance = job_input.get("face_enhance", False)
    bgr = job_input.get("bgr", False)
    
    if not image_source:
        return {"error": "❌ ข้อมูลตัวอินพุตสำคัญ 'image' ขาดหายไป"}
        
    if model_name not in MODELS:
        return {"error": f"❌ ไม่รู้จักชื่อโมเดลตัวเลือกนี้ กรุณาเลือกจาก: {list(MODELS.keys())}"}
        
    # 1. จัดการบูตตัวโหลดประมวลผลหลักอิงตาม Payload
    print(f"⚙️ การจัดเตรียมเครื่องยนต์หลัก: โหลดน้ำหนัก {model_name}...")
    upsampler, scale = get_upsampler(model_name)
    
    face_enhancer = None
    if face_enhance:
        print("🎭 [โหมดแต่งใบหน้าพรีเมียม] ตรวจพบเปิดใช้งานระบบ GFPGAN โหลดโมเดลแต่งหน้าชัด...")
        face_enhancer = get_face_enhancer(upsampler, scale)
        
    # 2. จัดสร้างโฟลเดอร์ทำงานจำลองในการดาวน์โหลด/แกะข้อมูล
    tmp_dir = tempfile.mkdtemp()
    input_path = os.path.join(tmp_dir, "input_img.png")
    
    # วิเคราะห์ประเภทข้อมูลภาพขาเข้า
    if image_source.startswith("http://") or image_source.startswith("https://"):
        print(f"📥 [Web Fetch] ทำการดึงไฟล์ต้นฉบับผ่านเครือข่ายอินเทอร์เน็ต: {image_source}")
        urllib.request.urlretrieve(image_source, input_path)
        img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    elif os.path.exists(image_source):
        print(f"📂 [Local Load] ดึงไฟล์ภาพโดยตรงจากเครื่องของคุณ: {image_source}")
        img = cv2.imread(image_source, cv2.IMREAD_COLOR)
    else:
        # พยายามถอดรหัสรูปแบบ Base64
        try:
            print("🔑 [Base64 Decode] ดำเนินการแกะแปลงภาพจากข้อมูลเข้ารหัส Base64...")
            if "," in image_source:
                image_source = image_source.split(",")[1]
            img_data = base64.b64decode(image_source)
            nparr = np.frombuffer(img_data, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        except Exception as e:
            return {"error": f"❌ ภาพต้นทางไม่ถูกต้อง ต้องเป็น URL, Local Path หรือ Base64: {str(e)}"}
            
    if img is None:
        return {"error": "❌ ไม่สามารถอ่านหรือแปลงพิกเซลจากรูปภาพที่ป้อนเข้ามาได้"}
        
    # 3. เริ่มสเต็ปรันการคำนวณอัปสเกล AI
    print("⚡ กำลังส่งพิกเซลภาพเข้าหาหน่วยประมวลผล GPU/CPU AI... ")
    h, w = img.shape[:2]
    start_time = cv2.getTickCount()
    
    try:
        # หากเปิดใช้งานแต่งใบหน้าคมชัดด้วย GFPGAN
        if face_enhance and face_enhancer is not None:
            # GFPGAN ต้องการรูปในแบบ BGR เป็นมาตรฐาน และจะผสาน Real-ESRGAN ขยายพื้นหลังให้อัตโนมัติ
            _, _, output = face_enhancer.enhance(img, has_aligned=False, only_center_face=False, paste_back=True)
        else:
            # การอัปสเกลทั่วไปด้วย Real-ESRGAN
            if bgr:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                output, _ = upsampler.enhance(img_rgb, outscale=scale)
                output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)
            else:
                output, _ = upsampler.enhance(img, outscale=scale)
                
        elapsed = (cv2.getTickCount() - start_time) / cv2.getTickFrequency()
        oh, ow = output.shape[:2]
        print(f"✅ การประมวลผลเสร็จสมบูรณ์! ใช้เวลาเพียง: {elapsed:.2f} วินาที")
        
        # 4. บันทึกผลลัพธ์และเซฟแปลงคืนเป็น Base64 กลับออกไปใน JSON
        out_path = os.path.join(tmp_dir, "output_img.png")
        cv2.imwrite(out_path, output, [cv2.IMWRITE_PNG_COMPRESSION, 3])
        
        with open(out_path, "rb") as f:
            b64_out = base64.b64encode(f.read()).decode("utf-8")
            
        return {
            "status": "success",
            "processing_time_sec": elapsed,
            "image": f"data:image/png;base64,{b64_out}",
            "image_format": "png",
            "model": model_name,
            "face_enhance_applied": face_enhance,
            "scale": scale,
            "input_size": {"width": w, "height": h},
            "output_size": {"width": ow, "height": oh}
        }
    except Exception as e:
        return {"error": f"❌ ล้มเหลวระหว่างขั้นตอนประมวลผล AI: {str(e)}"}
        
print("🧪 ตัว Handler จำลองระบบ API ของ RunPod Serverless พร้อมทำงาน E2E!")

### 📸 สเต็ปที่ 4: การรันการทดสอบและแสดงผลเปรียบเทียบ Before & After
รันเซลล์นี้เพื่อเริ่มต้นทดลองรันตัวประมวลผลจำลอง ด้วยรูปภาพบุคคลทดสอบ (หรือ URL อื่นๆ ที่คุณป้อน) เพื่อดูภาพแสดงผลเปรียบเทียบ Before & After ได้เห็นพลังแต่งหน้าคมชัดคมกริบทันตากันเลยครับ!

In [ ]:
#@title 🚀 4) กดปุ่มรันเซลล์นี้เพื่อเริ่มต้นทดสอบประมวลผลจริง { display-mode: "form" }
import matplotlib.pyplot as plt
from PIL import Image
import io
import tempfile

# ─── ⚙️ กำหนดค่าภาพทดลองและพารามิเตอร์ ───
# ภาพ Portrait สาวสวยดั้งเดิมอย่างเป็นทางการของทีมพัฒนา GFPGAN สำหรับรันเปรียบเทียบประสิทธิภาพ
TEST_IMAGE_URL = "https://raw.githubusercontent.com/TencentARC/GFPGAN/master/inputs/whole_imgs/00.jpg" #@param {type:"string"}
TEST_MODEL = "x2plus" #@param ["x2plus", "x4plus", "ultrasharp", "anime"]
APPLY_FACE_ENHANCE = True #@param {type:"boolean"}

print("🏁 เริ่มกระบวนการส่ง Payload จำลองประมวลผลภาพ E2E...")

# จำลองการสร้าง JSON Request จาก Next.js ของเรา
mock_payload = {
    "input": {
        "image": TEST_IMAGE_URL,
        "model": TEST_MODEL,
        "face_enhance": APPLY_FACE_ENHANCE
    }
}

# รันจำลอง API
result = runpod_mock_handler(mock_payload)

if "error" in result:
    print(f"\n❌ ล้มเหลว! ได้รับข้อความผิดพลาด: {result['error']}")
else:
    print(f"\n🎉 ประมวลผลสำเร็จและตอบกลับ JSON มาแล้ว!")
    print(f"⏱️ เวลาที่เครื่อง GPU ใช้: {result['processing_time_sec']:.3f} วินาที")
    print(f"📐 ขนาดภาพเดิม: {result['input_size']['width']}x{result['input_size']['height']} พิกเซล")
    print(f"📐 ขนาดหลังขยาย: {result['output_size']['width']}x{result['output_size']['height']} พิกเซล (ขยาย x{result['scale']})")
    print(f"🎭 สถานะการแต่งใบหน้าชัด: {'เปิดใช้งาน (GFPGAN Active)' if result['face_enhance_applied'] else 'ปิดใช้งาน'}")
    
    # ดำเนินการถอดพิกเซลภาพมาพล็ตเปรียบเทียบ
    tmp_dir = tempfile.mkdtemp()
    test_in_path = os.path.join(tmp_dir, "temp_test_in.jpg")
    urllib.request.urlretrieve(TEST_IMAGE_URL, test_in_path)
    
    img_original = Image.open(test_in_path)
    
    # แปลง Base64 เอาต์พุตกลับมาเป็นภาพจริง
    b64_img_data = result["image"].split(",")[1]
    output_bytes = base64.b64decode(b64_img_data)
    img_output = Image.open(io.BytesIO(output_bytes))
    
    # ─── 🎨 ทำการพล็อตแสดงผลพรีวิวเปรียบเทียบระดับพรีเมียม ───
    fig, axes = plt.subplots(1, 2, figsize=(16, 9))
    
    # 1. พล็อตภาพ Before
    axes[0].imshow(img_original)
    axes[0].set_title(f"Before (Original Portrait) \n {img_original.size[0]}x{img_original.size[1]} px", fontsize=13, fontweight='light', color='gray')
    axes[0].axis('off')
    
    # 2. พล็อตภาพ After
    axes[1].imshow(img_output)
    title_prefix = "After (AI Premium Upscaled)"
    if result["face_enhance_applied"]:
        title_prefix += "\n+ GFPGANv1.3 Face Enhancement HD"
    axes[1].set_title(f"{title_prefix} \n {img_output.size[0]}x{img_output.size[1]} px (x{result['scale']})", fontsize=13, fontweight='bold', color='#10b981')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n🎉 ทำงานและแสดงผล Before & After เปรียบเทียบเรียบร้อยสมบูรณ์ระดับพรีเมียมครับ! ✨")